# Direction 2 · Phase C · Step 5: Causal Block Patching, Branch Drift & `Affine-DFR` Audit (Kaggle GPU)

**Settings:** Enable **GPU T4 x1** and **Internet ON**.

**Zero manual uploads required:**
- Clones `spurious_correlation_mechanisms` from GitHub.
- Auto-downloads the 7 saved ResNet-50 `ckpt_last.pt` backbones, Waterbirds images (~490 MB), and Caltech `CUB_200_2011` pixel masks (~39 MB).
- Runs strictly on validation (`split == 1`, zero test access):
  - **Part A:** Spatially resolved causal activation patching (`bg_only`, `fg_only`, `full`) across `layer1..layer4.2` on the 26 persistent failures;
  - **Part B:** Bottleneck skip vs. 3x3 residual-conv drift & foreground ReLU gate-flip decomposition;
  - **Part C:** Stage-scoped BN-affine ablations (`bn_l1_l2`, `bn_l3`, `bn_l4`, `bn_l3_l4`, `bn_all`) + `Affine-DFR` counterfactual accuracy & Layer-4 foreground drift audit;
  - **Part D:** Channel bleed sensitivity `S_c` vs. mask-free observational group shift `G_c` (Spearman correlation).
- Outputs `/kaggle/working/rep_repair_phase_c_step5.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/kaggle/working/phase_c_assets')
OUT_DIR = Path('/kaggle/working/rep_repair_phase_c_step5')

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Pre-flight unit tests.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_phase_c_patching.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_phase_c_patching',
    '--work-dir', str(WORK_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'step5_causal_localization.png')))

zip_path = Path('/kaggle/working/rep_repair_phase_c_step5.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved results bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')